# Lab 06 — Support Vector Machine (SVM)
**Học máy ứng dụng · Nguyễn Hoàng Hải · 2474802010099**

Notebook bám theo tài liệu Lab06_SVM.pdf. Chạy lần lượt từ trên xuống. Bộ dữ liệu CSV được tìm ở thư mục hiện tại hoặc `data/`.

## 5. Gamma và dò lưới

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

def doc_csv(ten):
    ung_vien = [Path('data') / ten, Path(ten)]
    if ten == 'sinh_vien.csv':
        ung_vien += [Path('sinh_vien (1).csv'), Path('data/sinh_vien (1).csv')]
    for p in ung_vien:
        if p.exists():
            print(f'Đọc dữ liệu: {p}')
            return pd.read_csv(p)
    raise FileNotFoundError(f'Không tìm thấy {ten}. Đã tìm: {ung_vien}')

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
df=doc_csv('hai_vang_trang.csv')
X=df[['x1','x2']]; y=df['nhan']
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=.25,random_state=17,stratify=y)
for gamma in [.1,1,10,100]:
    m=SVC(kernel='rbf',gamma=gamma).fit(X_tr,y_tr)
    print(f'gamma={gamma:5g} train={m.score(X_tr,y_tr):.4f} test={m.score(X_te,y_te):.4f} support={len(m.support_)}')
search=GridSearchCV(SVC(kernel='rbf'),{'C':[.1,1,10,100],'gamma':[.01,.1,1,10]},cv=5)
search.fit(X_tr,y_tr)
scores=pd.DataFrame(search.cv_results_['params']); scores['cv_score']=search.cv_results_['mean_test_score']
print('Điểm CV (hàng=C, cột=gamma):')
display(scores.pivot(index='C',columns='gamma',values='cv_score').round(4))
print('Tham số tốt nhất:',search.best_params_)
print(f'CV tốt nhất={search.best_score_:.4f}; test={search.score(X_te,y_te):.4f}')

**Nhận xét:** Gamma nhỏ làm ảnh hưởng mỗi vector hỗ trợ lan rộng, gamma quá lớn dễ tạo biên gồ ghề và overfit. Chọn đồng thời C và gamma bằng kiểm định chéo trên tập huấn luyện; tập kiểm tra chỉ dùng đánh giá cuối.